# Notebook 22 — v2 Validation, Sensitivity, Submission

**Purpose:** the safety-net step. Runs the 6-item auto-validation, sensitivity sweep, top-100 manual audits, builds the DAG, and writes a submission CSV with the **correct** column name (`Outlet_ID`) and **20,000 rows**.

**What changed vs `04_model_validation.ipynb`:**

- V1 expects `Outlet_ID` column (not `row_id` — per official PDF).
- V1 expects 20,000 rows (not 914 — the 914-row constraint is unverified).
- V5 cap-binding uses bucket-specific `cap_uplift` (not hardcoded 5.9×).
- All numbers come from the v2 modeling notebook (notebook 21).

**Submission policy (Option A — non-destructive):**

- Writes to `Results/smil_labs_predictions_v2.csv` (does NOT overwrite `smil_labs_predictions.csv`).
- You can compare side-by-side and choose which to upload.
- The team's existing `smil_labs_predictions.csv` is NOT touched by this notebook.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "Notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.reporting import build_dag, run_validation_suite, sensitivity_sweep

GOLD_DIR = ROOT / "data" / "gold"
SILVER_DIR = ROOT / "data" / "silver"
RESULTS_DIR = ROOT / "Results"
REPORTS_DIR = ROOT / "Reports"
(REPORTS_DIR / "figures").mkdir(parents=True, exist_ok=True)

preds = pd.read_parquet(GOLD_DIR / "predictions_v2.parquet")
quantile_preds = pd.read_parquet(GOLD_DIR / "quantile_predictions_v2.parquet")
cap_table = pd.read_csv(GOLD_DIR / "cap_table_v2.csv")
gold = pd.read_parquet(GOLD_DIR / "outlet_features.parquet")
transactions = pd.read_parquet(SILVER_DIR / "transactions_history.parquet")
outlet_master = pd.read_parquet(SILVER_DIR / "outlet_master.parquet")

print(f"Loaded {len(preds)} predictions, {len(cap_table)} cap rows.")


## 1 — Write the v2 submission CSV (Outlet_ID + 20,000 rows)

In [ ]:
sub = preds[["Outlet_ID", "Maximum_Monthly_Liters"]].copy()
sub["Maximum_Monthly_Liters"] = sub["Maximum_Monthly_Liters"].round(3)

# Sanity guards before writing
assert "Outlet_ID" in sub.columns, "FATAL: column must be Outlet_ID per official PDF"
assert sub["Outlet_ID"].is_unique, "FATAL: duplicate Outlet_IDs"
assert not sub["Maximum_Monthly_Liters"].isna().any(), "FATAL: NaN in predictions"
assert (sub["Maximum_Monthly_Liters"] >= 0).all(), "FATAL: negative predictions"

# Reindex to outlet_master order (deterministic)
sub = outlet_master[["Outlet_ID"]].merge(sub, on="Outlet_ID", how="left")
print(f"Submission shape: {sub.shape}")
print(f"Columns: {sub.columns.tolist()}")

sub_path = RESULTS_DIR / "smil_labs_predictions_v2.csv"
sub.to_csv(sub_path, index=False)

# Also save a full-20k version for parity with team naming
sub.to_csv(RESULTS_DIR / "smil_labs_predictions_full_20000_v2.csv", index=False)
print(f"\nWrote {sub_path}")
print(f"Wrote {RESULTS_DIR / 'smil_labs_predictions_full_20000_v2.csv'}")

sub.head()


## 2 — Run the 6-item auto-validation suite

In [ ]:
historical_max = gold.set_index("Outlet_ID")["observed_max_monthly_liters"]
bucket_keys = gold[["Outlet_ID", "Outlet_Type", "Outlet_Size"]].copy()

result = run_validation_suite(
    submission=sub,
    outlet_master=outlet_master,
    historical_max=historical_max,
    cap_table=cap_table,
    bucket_keys=bucket_keys,
    out_dir=RESULTS_DIR,
)
print(f"All passed: {result.all_passed}\n")
for c in result.checks:
    mark = "OK  " if c.passed else "FAIL"
    print(f"  [{mark}] {c.name} -- {c.detail}")


## 3 — Sensitivity sweep on free knobs

Sweeps frontier quantile × constraint-score weighting × cap multiplier. Output -> `Reports/figures/sensitivity_table.csv`.

In [ ]:
lower_bounds = preds[["Outlet_ID", "lower_bound"]].copy()
sens = sensitivity_sweep(
    features=gold,
    transactions=transactions,
    lower_bounds=lower_bounds,
    multi_q_predictions=quantile_preds,
    cap_table=cap_table,
    out_dir=REPORTS_DIR / "figures",
)
sens.head(20)


## 4 — DAG figure

In [ ]:
dag_paths = build_dag(REPORTS_DIR / "figures")
print(f"DAG outputs:")
for k, v in dag_paths.items():
    print(f"  {k}: {v}")


## 5 — Top-100 highest-potential audit (hostile-judge sanity)

In [ ]:
top100_potential = preds.nlargest(100, "Maximum_Monthly_Liters")[
    ["Outlet_ID", "Outlet_Type", "Outlet_Size", "observed_max_monthly_liters",
     "lower_bound", "frontier_q90", "constraint_score", "Maximum_Monthly_Liters", "uplift_ratio"]
]
top100_potential.to_csv(GOLD_DIR / "validation_top_100_potential_v2.csv", index=False)

# Sanity: top100 should be biased toward Large/Extra Large outlets
print("Top-100 potential by Outlet_Size:")
print(top100_potential["Outlet_Size"].value_counts())
print()
print("Top-100 potential by Outlet_Type:")
print(top100_potential["Outlet_Type"].value_counts())
top100_potential.head(10)


## 6 — Top-100 highest-uplift audit (anti-overfit check)

In [ ]:
top100_uplift = preds.nlargest(100, "uplift_ratio")[
    ["Outlet_ID", "Outlet_Type", "Outlet_Size", "observed_max_monthly_liters",
     "lower_bound", "frontier_q90", "constraint_score", "Maximum_Monthly_Liters", "uplift_ratio"]
]
top100_uplift.to_csv(GOLD_DIR / "validation_top_100_uplift_v2.csv", index=False)

# Sanity: high uplifts should come from outlets with high constraint_score, NOT just any outlet
print(f"Top-100 uplift mean constraint_score: {top100_uplift['constraint_score'].mean():.3f}")
print(f"Population mean constraint_score: {preds['constraint_score'].mean():.3f}")
print(f"  (top-100 should be >> population mean)")
print()
print("Top-100 uplift by Outlet_Size:")
print(top100_uplift["Outlet_Size"].value_counts())
top100_uplift.head(10)


## 7 — Comparison vs the team's v1 submission

Side-by-side: how does the v2 model differ from `smil_labs_predictions.csv`?

In [ ]:
v1_path = RESULTS_DIR / "smil_labs_predictions.csv"
if v1_path.exists():
    v1 = pd.read_csv(v1_path)
    # v1 might use row_id; normalise
    v1 = v1.rename(columns={"row_id": "Outlet_ID"})
    print(f"v1 shape: {v1.shape} (was 914 rows in original)")
    print(f"v1 columns: {v1.columns.tolist()}")
    print()
    overlap = sub.merge(v1, on="Outlet_ID", how="inner", suffixes=("_v2", "_v1"))
    print(f"Outlets present in BOTH v1 and v2: {len(overlap)}")
    if len(overlap):
        delta = overlap["Maximum_Monthly_Liters_v2"] - overlap["Maximum_Monthly_Liters_v1"]
        print(f"\nv2 - v1 prediction delta (over the {len(overlap)} overlapping outlets):")
        print(f"  median: {delta.median():.2f}")
        print(f"  mean:   {delta.mean():.2f}")
        print(f"  pct higher in v2: {(delta > 0).mean() * 100:.1f}%")
        print(f"  pct lower in v2:  {(delta < 0).mean() * 100:.1f}%")
else:
    print(f"No v1 submission found at {v1_path} (skipping comparison).")


## Summary

- Submission: `Results/smil_labs_predictions_v2.csv` with `Outlet_ID, Maximum_Monthly_Liters` and 20,000 rows.
- Validation: 6 auto checks (V1 schema, V2 NaN/neg/dup, V3a IDs in master, V3b ≥ historical max for ≥99%, V4 median uplift in range, V5 cap-binding rate using bucket-specific cap).
- Sensitivity: knob-sweep table at `Reports/figures/sensitivity_table.csv`.
- DAG: at `Reports/figures/dag.{png,mmd}`.
- Manual audits: top-100 by potential and top-100 by uplift, both at `data/gold/`.
- Comparison to v1 printed above.

**To submit the v2 file:**

```powershell
copy Results\smil_labs_predictions_v2.csv Results\smil_labs_predictions.csv
```

(or upload `smil_labs_predictions_v2.csv` directly).

**Before you submit, verify with the portal whether the 914-row constraint is real.** If the portal accepts only 914 rows, filter `smil_labs_predictions_v2.csv` to whatever `Outlet_ID` list the portal expects.
